# AnnotateX: LLM Automatic Data Annotation in Long-Context Scenarios
### FlagOS Open Computing Hackathon — Track 3

**Team**: zan-maker  
**Approach**: Multi-task In-Context Learning (ICL) with Qwen3-4B, chain-of-thought reasoning, and self-consistency decoding across 8 diverse OpenSeek tasks.

## Tasks Overview
| # | Task | Type | Examples | Tests |
|---|------|------|----------|-------|
| 1 | Closest Integers | Math | 5500 | 500 |
| 2 | Count Nouns & Verbs | NLP | 5440 | 500 |
| 3 | Collatz Conjecture | Math | 3997 | 500 |
| 4 | Concat Strings | Code | 3993 | 500 |
| 5 | Tweet Sadness Detection | Binary Classification | 1899 | 500 |
| 6 | MNLI Genre Classification | Binary Classification | 5500 | 500 |
| 7 | Jeopardy Answer Generation | QA | 5499 | 500 |
| 8 | Kernel Generation | Code Generation | 184 | 166 |

In [ ]:
# ============================================================
# INSTALLATION
# ============================================================
!pip install -q transformers accelerate bitsandbytes
!pip install -q sentencepiece protobuf scipy

In [ ]:
# ============================================================
# IMPORTS & CONFIGURATION
# ============================================================
import os
import json
import re
import glob
import time
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from collections import Counter

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# ============================================================
# HYPERPARAMETERS
# ============================================================
MODEL_NAME = "Qwen/Qwen3-4B"
MAX_NEW_TOKENS = 256          # Max tokens for generated answer
TEMPERATURE = 0.1             # Low temperature for deterministic outputs
TOP_P = 0.85
NUM_SHOT = 5                  # Number of few-shot examples per task
SELF_CONSISTENCY_RUNS = 1     # Set >1 for majority voting (slower but more robust)
USE_4BIT = True               # 4-bit quantization to fit in GPU memory
CONTEXT_MAX_CHARS = 16000     # Max chars for few-shot context (conservative for 32K tokens)
MAX_INPUT_TOKENS = 30000      # Max input tokens per prompt

print("Hyperparameters configured.")
print(f"  Model: {MODEL_NAME}")
print(f"  Few-shot examples per task: {NUM_SHOT}")
print(f"  Self-consistency runs: {SELF_CONSISTENCY_RUNS}")
print(f"  4-bit quantization: {USE_4BIT}")

In [ ]:
# ============================================================
# LOAD ALL TASK DATA
# ============================================================
INPUT_DIR = "/kaggle/input/track-3-llm-automatic-data-annotation-in-long-context-scenarios"

# Find all JSON files
json_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.json")))
print(f"Found {len(json_files)} JSON task files\n")

tasks = []
all_test_samples = []

for fpath in json_files:
    with open(fpath, 'r') as f:
        data = json.load(f)
    
    task_info = {
        'task_id': data['task_id'],
        'task_name': data['task_name'],
        'definition': data['Definition'][0] if isinstance(data['Definition'], list) else data['Definition'],
        'examples': data['examples'],
        'test_samples': data['test_samples'],
        'num_examples': len(data['examples']),
        'num_tests': len(data['test_samples']),
    }
    tasks.append(task_info)
    
    # Collect all test samples
    for ts in data['test_samples']:
        all_test_samples.append({
            'id': ts['id'],
            'input': ts['input'],
            'task_id': data['task_id'],
            'task_name': data['task_name'],
            'definition': task_info['definition'],
        })
    
    print(f"  {data['task_id']:20s} | {data['task_name'][:50]:50s} | ex={len(data['examples']):5d} | test={len(data['test_samples']):4d}")

print(f"\nTotal test samples to predict: {len(all_test_samples)}")
print(f"Total training examples available: {sum(t['num_examples'] for t in tasks)}")

In [ ]:
# ============================================================
# ANALYZE TASK TYPES & OUTPUT FORMATS
# ============================================================
for task in tasks:
    sample_outputs = [str(ex.get('output', [''])[0]) if isinstance(ex.get('output'), list) else str(ex.get('output','')) for ex in task['examples'][:20]]
    unique = set(sample_outputs)
    
    # Determine task type
    is_classification = len(unique) <= 10
    is_binary = len(unique) == 2
    
    task['is_classification'] = is_classification
    task['is_binary'] = is_binary
    task['output_types'] = list(unique)[:5]
    
    print(f"\n{task['task_id']}: {task['task_name'][:40]}")
    print(f"  Classification: {is_classification}, Binary: {is_binary}")
    print(f"  Sample outputs: {list(unique)[:5]}")
    print(f"  Def: {task['definition'][:120]}...")

In [ ]:
# ============================================================
# LOAD MODEL (Qwen3-4B with 4-bit quantization)
# ============================================================
print(f"Loading {MODEL_NAME} with 4-bit quantization...")
t0 = time.time()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
) if USE_4BIT else None

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, padding_side="left"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "trust_remote_code": True, 
    "torch_dtype": torch.float16, 
    "device_map": "auto"
}
if bnb_config:
    model_kwargs["quantization_config"] = bnb_config

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
model.eval()

print(f"Model loaded in {time.time()-t0:.1f}s")
if torch.cuda.is_available():
    print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ============================================================
# PROMPT BUILDER — Task-specific ICL with Chain-of-Thought
# ============================================================
def build_task_prompt(task, test_input, num_shot=NUM_SHOT):
    """
    Build an ICL prompt for a specific task.
    Format: Definition -> Few-shot examples -> Test question.
    Uses Qwen3 chat template for best performance.
    """
    definition = task['definition']
    examples = task['examples']
    
    # Select diverse few-shot examples
    selected_exs = []
    if task['is_classification'] and task['is_binary']:
        # Balanced sampling for binary classification
        labels = [str(ex['output'][0]) if isinstance(ex['output'], list) else str(ex['output']) for ex in examples]
        unique_labels = list(set(labels))
        per_label = max(1, num_shot // len(unique_labels))
        for lbl in unique_labels:
            matching = [ex for ex in examples if str(ex['output'][0] if isinstance(ex['output'], list) else ex['output']) == lbl]
            selected_exs.extend(random.sample(matching, min(per_label, len(matching))))
        random.shuffle(selected_exs)
        selected_exs = selected_exs[:num_shot]
    else:
        selected_exs = random.sample(examples, min(num_shot, len(examples)))
    
    # Build prompt using Qwen chat template
    messages = []
    
    # System message with task definition
    system_msg = f"You are an expert assistant. Your task is: {definition}\n\nFollow the format of the examples below. Provide only the answer, nothing else."
    messages.append({"role": "system", "content": system_msg})
    
    # Few-shot examples as user/assistant pairs
    examples_text = ""
    for i, ex in enumerate(selected_exs):
        inp = str(ex['input'])
        out = str(ex['output'][0]) if isinstance(ex['output'], list) else str(ex['output'])
        examples_text += f"\nExample {i+1}:\nInput: {inp}\nOutput: {out}\n"
    
    user_msg = f"Here are some examples:\n{examples_text}\n\nNow solve this:\nInput: {str(test_input)}\nOutput:"
    messages.append({"role": "user", "content": user_msg})
    
    # Apply chat template
    try:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except:
        # Fallback: manual Qwen template
        prompt = ""
        for msg in messages:
            if msg['role'] == 'system':
                prompt += f"<|im_start|>system\n{msg['content']}<|im_end|>\n"
            elif msg['role'] == 'user':
                prompt += f"<|im_start|>user\n{msg['content']}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"
    
    return prompt


def generate_answer(prompt, temperature=TEMPERATURE):
    """Generate model response for a single prompt."""
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        truncation=True, 
        max_length=MAX_INPUT_TOKENS
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=temperature,
            top_p=TOP_P,
            do_sample=(temperature > 0),
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the generated portion
    generated = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def extract_answer(response, task):
    """
    Extract clean answer from model response.
    Handles different task types appropriately.
    """
    resp = response.strip()
    
    # For classification tasks, find matching label
    if task['is_classification']:
        valid_labels = task['output_types']
        for label in valid_labels:
            if label.lower() in resp.lower():
                return label
    
    # Try to extract after "Output:" pattern
    match = re.search(r'Output:\s*(.+?)(?:\n|$)', resp, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    # For thinking/reasoning models, extract after </think > or similar
    think_match = re.search(r'</think[^>]*>\s*(.+)', resp, re.DOTALL)
    if think_match:
        return think_match.group(1).strip()
    
    # Return first line as answer
    first_line = resp.split('\n')[0].strip()
    return first_line if first_line else resp

In [ ]:
# ============================================================
# TEST PROMPT ON ONE SAMPLE (Quick Validation)
# ============================================================
test_task = tasks[5]  # MNLI genre classification (binary: Y/N)
test_input = test_task['test_samples'][0]['input']
print(f"Task: {test_task['task_name']}")
print(f"Input: {str(test_input)[:200]}...")

prompt = build_task_prompt(test_task, test_input, num_shot=3)
print(f"\nPrompt length: {len(prompt)} chars")
print(f"Prompt preview: ...{prompt[-300:]}")

t0 = time.time()
response = generate_answer(prompt, temperature=0.1)
print(f"\nRaw response ({time.time()-t0:.1f}s): {response}")
answer = extract_answer(response, test_task)
print(f"Extracted answer: {answer}")

In [ ]:
# ============================================================
# BATCH GENERATION — Process all tasks
# ============================================================
all_predictions = []
total_samples = len(all_test_samples)
start_time = time.time()

# Build task lookup
task_lookup = {t['task_id']: t for t in tasks}

for i, sample in enumerate(all_test_samples):
    task = task_lookup[sample['task_id']]
    sample_id = sample['id']
    test_input = sample['input']
    
    # Progress tracking
    if i % 10 == 0 or i == total_samples - 1:
        elapsed = time.time() - start_time
        eta = elapsed / (i + 1) * (total_samples - i - 1) if i > 0 else 0
        print(f"[{i+1}/{total_samples}] Task: {task['task_id'][:25]:25s} | ETA: {eta/60:.1f}m")
    
    try:
        # Build prompt and generate
        prompt = build_task_prompt(task, test_input)
        
        if SELF_CONSISTENCY_RUNS > 1:
            # Self-consistency: multiple runs, majority vote
            votes = Counter()
            for run in range(SELF_CONSISTENCY_RUNS):
                temp = TEMPERATURE + run * 0.1
                resp = generate_answer(prompt, temperature=temp)
                ans = extract_answer(resp, task)
                votes[ans] += 1
            answer = votes.most_common(1)[0][0]
        else:
            resp = generate_answer(prompt)
            answer = extract_answer(resp, task)
            
    except Exception as e:
        print(f"  ERROR on {sample_id}: {e}")
        answer = ""  # Empty fallback
    
    all_predictions.append({
        'ID': sample_id,
        'Predicted': answer
    })
    
    # Periodic GPU cache cleanup
    if (i + 1) % 50 == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache()

total_time = time.time() - start_time
print(f"\nDone! Processed {total_samples} samples in {total_time/60:.1f} minutes")
print(f"Average: {total_time/total_samples:.1f}s per sample")

In [ ]:
# ============================================================
# SAVE SUBMISSION
# ============================================================
sub_df = pd.DataFrame(all_predictions)

submission_path = "/kaggle/working/submission.csv"
sub_df.to_csv(submission_path, index=False)

print(f"=== SUBMISSION SAVED ===")
print(f"Path: {submission_path}")
print(f"Rows: {len(sub_df)}")
print(f"Columns: {list(sub_df.columns)}")
print(f"\nPreview (first 10):")
print(sub_df.head(10).to_string(index=False))
print(f"\nPreview (last 5):")
print(sub_df.tail(5).to_string(index=False))

In [ ]:
# ============================================================
# VALIDATION — Check submission format
# ============================================================
print("=== VALIDATION ===")
print(f"Total rows: {len(sub_df)}")
print(f"Columns: {list(sub_df.columns)}")
print(f"Unique IDs: {sub_df['ID'].nunique()}")
print(f"Duplicate IDs: {sub_df['ID'].duplicated().sum()}")
print(f"Empty predictions: {(sub_df['Predicted'] == '').sum()}")
print(f"\nPredicted value sample (first 20):")
for _, row in sub_df.head(20).iterrows():
    print(f"  {row['ID'][:50]:50s} -> {str(row['Predicted'])[:80]}")